In [24]:
from workspace.sources.local_datasets.preprocessing.tokenization import NLTKTokenizer
from workspace.sources.local_datasets.preprocessing.cleaning import NoiseReduction, Lemmatization, Stemming
from workspace.sources.local_datasets.preprocessing.encoders.encoders import BertBaseUncasedEncoder
from workspace.sources.experiments.metrics import Loss
from workspace.sources.experiments.experiment import Experiment
from workspace.sources.local_datasets.dataset import Dataset
from workspace.sources.local_datasets.preprocessing.pipelines import PreprocessingPipeline
from workspace.sources.models.transformers.bert_based_models import BERT
from workspace.sources.local_datasets.preprocessing.utils import truncate
from sklearn.model_selection import ParameterGrid
from IPython.display import display, Markdown
import mlflow

mlflow.set_tracking_uri('../../mlruns')

In [8]:
experiment_name = 'prefinal-bert-base-uncased-v2'
model = BERT
encoder = BertBaseUncasedEncoder

In [19]:
def conduct_model_experiments(dataset_name,
                              dataset_path,
                              dataset_hparams_grid,
                              model_hparams_grid):
    total_runs = len(dataset_hparams_grid) * len(model_hparams_grid)
    for i, dataset_params in enumerate(dataset_hparams_grid):
        for j, model_hparams in enumerate(model_hparams_grid):
            run_number = i * len(model_hparams_grid) + j + 1
            display(Markdown(f'### Run {run_number} of {total_runs}'))
            dataset = Dataset(dataset_name, dataset_path, **dataset_params)

            model_ = model(train_best_model_metric=Loss,
                         training_arguments=model_hparams)

            recovery_experiment = Experiment(
                name=experiment_name,
                dataset=dataset,
                model=model_)
            recovery_experiment.run()


In [20]:

# max_tokens_params = [128, 512]
max_tokens_params = [128]

pipelines = []

for max_tokens in max_tokens_params:
    pipelines.extend([PreprocessingPipeline(name='minimal',
                                            iterable=[encoder(truncation_max_length=max_tokens)]),
                      PreprocessingPipeline(name='noise_reduction',
                                            iterable=[NoiseReduction(),
                                                      encoder(truncation_max_length=max_tokens)]),
                      PreprocessingPipeline(name='noise_reduction_with_lemmatization',
                                            iterable=[NoiseReduction(), NLTKTokenizer(), Lemmatization(),
                                                      encoder(is_split_into_words=True,
                                                                             truncation_max_length=max_tokens)]),
                      PreprocessingPipeline(name='noise_reduction_with_stemming',
                                            iterable=[NoiseReduction(), NLTKTokenizer(), Stemming(),
                                                      encoder(is_split_into_words=True,
                                                                             truncation_max_length=max_tokens)])
                      ])
# optional - for testing purposes, if you want to run fast test on very small datasets
truncate(pipelines, fraction=0.05)

dataset_hparams_grid = ParameterGrid({'preprocessings_pipeline': pipelines})

print('Dataset Hyperparameters Grid Size: ', len(dataset_hparams_grid))

# model_hparams_grid = ParameterGrid({'epochs': [10],
#                                     'batch_size': [16],
#                                     'learning_rate': [1e-05, 5e-05],
#                                     'lr_scheduler': ['linear'],
#                                     'optimizer': ['adamw_torch'],
#                                     'weight_decay': [0.0, 1e-3],
#                                     'early_stopping_patience': [3],
#                                     'early_stopping_threshold': [0.01]})

model_hparams_grid = ParameterGrid({'epochs': [10],
                                    'batch_size': [16],
                                    'learning_rate': [5e-05],
                                    'lr_scheduler': ['linear'],
                                    'optimizer': ['adamw_torch'],
                                    'weight_decay': [1e-3],
                                    'early_stopping_patience': [3],
                                    'early_stopping_threshold': [0.01]})

print('Model Hyperparameters Grid Size: ', len(model_hparams_grid))
print('Total Hyperparameter Combinations: ', len(dataset_hparams_grid) * len(model_hparams_grid))

Dataset Hyperparameters Grid Size:  4
Model Hyperparameters Grid Size:  1
Total Hyperparameter Combinations:  4


### ReCOVery Dataset

In [21]:
dataset_name = 'ReCOVery'
dataset_path = '../../sources/local_datasets/data/ReCOVery/recovery.csv'

conduct_model_experiments(dataset_name,
                          dataset_path,
                          dataset_hparams_grid,
                          model_hparams_grid)

### Run 1 of 4

2025-05-16 00:15:29,413 - Experiment - INFO - MLflow experiment initialized with ID: 237384892919634716
2025-05-16 00:15:29,413 - Experiment - INFO - Model signature: model(mn=bert,ti=bert-base-uncased,mmn=loss,e=10,bs=16,ebs=16,lr=5e-05,ls=linear,wd=0.001,o=adamw_torch,esp=3,est=0.01)
2025-05-16 00:15:29,415 - Experiment - INFO - Dataset signature: dataset(dn=recovery,tp=0.7,vp=0.15);pipeline([truncation(f=0.05,tt=top),bert_base_uncased_encoder(t=bert-base-uncased,t=true,p=max_length,tml=128,isiw=false)])
2025-05-16 00:15:29,416 - Experiment - INFO - Run signature: model(mn=bert,ti=bert-base-uncased,mmn=loss,e=10,bs=16,ebs=16,lr=5e-05,ls=linear,wd=0.001,o=adamw_torch,esp=3,est=0.01);dataset(dn=recovery,tp=0.7,vp=0.15);pipeline([truncation(f=0.05,tt=top),bert_base_uncased_encoder(t=bert-base-uncased,t=true,p=max_length,tml=128,isiw=false)])
2025-05-16 00:15:29,537 - Experiment - INFO - Found existing run with signature model(mn=bert,ti=bert-base-uncased,mmn=loss,e=10,bs=16,ebs=16,lr=5e

### Run 2 of 4

2025-05-16 00:15:29,556 - Experiment - INFO - MLflow experiment initialized with ID: 237384892919634716
2025-05-16 00:15:29,557 - Experiment - INFO - Model signature: model(mn=bert,ti=bert-base-uncased,mmn=loss,e=10,bs=16,ebs=16,lr=5e-05,ls=linear,wd=0.001,o=adamw_torch,esp=3,est=0.01)
2025-05-16 00:15:29,558 - Experiment - INFO - Dataset signature: dataset(dn=recovery,tp=0.7,vp=0.15);pipeline([truncation(f=0.05,tt=top),noise_reduction(),bert_base_uncased_encoder(t=bert-base-uncased,t=true,p=max_length,tml=128,isiw=false)])
2025-05-16 00:15:29,558 - Experiment - INFO - Run signature: model(mn=bert,ti=bert-base-uncased,mmn=loss,e=10,bs=16,ebs=16,lr=5e-05,ls=linear,wd=0.001,o=adamw_torch,esp=3,est=0.01);dataset(dn=recovery,tp=0.7,vp=0.15);pipeline([truncation(f=0.05,tt=top),noise_reduction(),bert_base_uncased_encoder(t=bert-base-uncased,t=true,p=max_length,tml=128,isiw=false)])
2025-05-16 00:15:29,688 - Experiment - INFO - Found existing run with signature model(mn=bert,ti=bert-base-unca

### Run 3 of 4

...

2025-05-16 00:16:43,841 - Experiment - INFO - Test metrics: {'test_loss': 0.745797872543335, 'test_accuracy': 0.2, 'test_precision': 0.0, 'test_recall': 0.0, 'test_f1_score': 0.0, 'test_roc_auc': 0.3928571428571429, 'test_false_positives_rate': 0.5714285714285714, 'test_false_negatives_rate': 1.0, 'test_runtime': 0.1368, 'test_samples_per_second': 109.614, 'test_steps_per_second': 7.308, 'test_epoch': 2.0}
2025-05-16 00:16:43,858 - Experiment - INFO - Saving evaluation data in pickle...
2025-05-16 00:16:43,860 - Experiment - INFO - Saving evaluation data in json...
2025-05-16 00:16:43,900 - Experiment - INFO - Successfully saved evaluation data.
2025-05-16 00:16:43,901 - Experiment - INFO - Evaluating model using f1_score metric...
2025-05-16 00:16:43,901 - Experiment - INFO - Found checkpoint with best metrics at: checkpoint-15
2025-05-16 00:16:44,326 - Experiment - INFO - Best entry according to validation metrics: {'eval_loss': 0.6527726054191589, 'eval_accuracy': 0.6, 'eval_precisi

2025-05-16 00:16:44,469 - Experiment - INFO - Test metrics: {'test_loss': 0.7644299864768982, 'test_accuracy': 0.5333333333333333, 'test_precision': 0.5384615384615384, 'test_recall': 0.875, 'test_f1_score': 0.6666666666666666, 'test_roc_auc': 0.44642857142857145, 'test_false_positives_rate': 0.8571428571428571, 'test_false_negatives_rate': 0.125, 'test_runtime': 0.1359, 'test_samples_per_second': 110.357, 'test_steps_per_second': 7.357, 'test_epoch': 3.0}
2025-05-16 00:16:44,485 - Experiment - INFO - Saving evaluation data in pickle...
2025-05-16 00:16:44,487 - Experiment - INFO - Saving evaluation data in json...
2025-05-16 00:16:44,532 - Experiment - INFO - Successfully saved evaluation data.
2025-05-16 00:16:44,533 - Experiment - INFO - Evaluating model using false_positives_rate metric...
2025-05-16 00:16:44,534 - Experiment - INFO - Found checkpoint with best metrics at: checkpoint-10
2025-05-16 00:16:44,972 - Experiment - INFO - Best entry according to validation metrics: {'eval

2025-05-16 00:16:45,110 - Experiment - INFO - Test metrics: {'test_loss': 0.745797872543335, 'test_accuracy': 0.2, 'test_precision': 0.0, 'test_recall': 0.0, 'test_f1_score': 0.0, 'test_roc_auc': 0.3928571428571429, 'test_false_positives_rate': 0.5714285714285714, 'test_false_negatives_rate': 1.0, 'test_runtime': 0.1316, 'test_samples_per_second': 113.954, 'test_steps_per_second': 7.597, 'test_epoch': 2.0}
2025-05-16 00:16:45,131 - Experiment - INFO - Saving evaluation data in pickle...
2025-05-16 00:16:45,133 - Experiment - INFO - Saving evaluation data in json...
2025-05-16 00:16:45,173 - Experiment - INFO - Successfully saved evaluation data.
2025-05-16 00:16:45,174 - Experiment - INFO - Evaluating model using roc_auc metric...
2025-05-16 00:16:45,174 - Experiment - INFO - Found checkpoint with best metrics at: checkpoint-5
2025-05-16 00:16:45,627 - Experiment - INFO - Best entry according to validation metrics: {'eval_loss': 0.7145585417747498, 'eval_accuracy': 0.4666666666666667, 

2025-05-16 00:16:45,768 - Experiment - INFO - Test metrics: {'test_loss': 0.7016640901565552, 'test_accuracy': 0.5333333333333333, 'test_precision': 0.5333333333333333, 'test_recall': 1.0, 'test_f1_score': 0.6956521739130435, 'test_roc_auc': 0.6071428571428571, 'test_false_positives_rate': 1.0, 'test_false_negatives_rate': 0.0, 'test_runtime': 0.134, 'test_samples_per_second': 111.906, 'test_steps_per_second': 7.46, 'test_epoch': 1.0}
2025-05-16 00:16:45,785 - Experiment - INFO - Saving evaluation data in pickle...
2025-05-16 00:16:45,788 - Experiment - INFO - Saving evaluation data in json...
2025-05-16 00:16:45,830 - Experiment - INFO - Successfully saved evaluation data.
2025-05-16 00:16:45,831 - Experiment - INFO - Evaluating model using loss metric...
2025-05-16 00:16:45,832 - Experiment - INFO - Found checkpoint with best metrics at: checkpoint-15
2025-05-16 00:16:46,248 - Experiment - INFO - Best entry according to validation metrics: {'eval_loss': 0.6527726054191589, 'eval_accu

2025-05-16 00:16:46,382 - Experiment - INFO - Test metrics: {'test_loss': 0.7644299864768982, 'test_accuracy': 0.5333333333333333, 'test_precision': 0.5384615384615384, 'test_recall': 0.875, 'test_f1_score': 0.6666666666666666, 'test_roc_auc': 0.44642857142857145, 'test_false_positives_rate': 0.8571428571428571, 'test_false_negatives_rate': 0.125, 'test_runtime': 0.13, 'test_samples_per_second': 115.377, 'test_steps_per_second': 7.692, 'test_epoch': 3.0}
2025-05-16 00:16:46,398 - Experiment - INFO - Saving evaluation data in pickle...
2025-05-16 00:16:46,399 - Experiment - INFO - Saving evaluation data in json...
2025-05-16 00:16:46,440 - Experiment - INFO - Successfully saved evaluation data.
2025-05-16 00:16:46,441 - Experiment - INFO - Finished model evaluations stage.


### EUvsDisinfo Dataset

In [ ]:
dataset_name = 'EUvsDisinfo'
dataset_path = '../../sources/local_datasets/data/EUvsDisinfo/euvsdisinfo.csv'

conduct_model_experiments(dataset_name,
                          dataset_path,
                          dataset_hparams_grid,
                          model_hparams_grid)

### ISOT Dataset

In [ ]:
dataset_name = 'ISOT'
dataset_path = '../../sources/local_datasets/data/ISOT/isot.csv'

conduct_model_experiments(dataset_name,
                          dataset_path,
                          dataset_hparams_grid,
                          model_hparams_grid)

### CZ Dataset

In [22]:
cz_pipelines = []

for max_tokens in max_tokens_params:
    cz_pipelines.extend([PreprocessingPipeline(name='minimal',
                                               iterable=[BertBaseUncasedEncoder(truncation_max_length=max_tokens)]),
                         PreprocessingPipeline(name='noise_reduction',
                                               iterable=[NoiseReduction(),
                                                         BertBaseUncasedEncoder(truncation_max_length=max_tokens)]),
                         PreprocessingPipeline(name='noise_reduction_with_lemmatization',
                                               iterable=[NoiseReduction(), NLTKTokenizer(language='czech'),
                                                         Lemmatization(language='czech'),
                                                         BertBaseUncasedEncoder(is_split_into_words=True,
                                                                                truncation_max_length=max_tokens)]),
                         PreprocessingPipeline(name='noise_reduction_with_stemming',
                                               iterable=[NoiseReduction(), NLTKTokenizer(language='czech'),
                                                         Stemming(language='czech'),
                                                         BertBaseUncasedEncoder(is_split_into_words=True,
                                                                                truncation_max_length=max_tokens)])
                         ])
# optional - for testing purposes, if you want to run fast test on very small datasets
truncate(cz_pipelines, fraction=0.05)

cz_dataset_hparams_grid = ParameterGrid({'preprocessings_pipeline': cz_pipelines})
print('Dataset Hyperparameters Grid Size: ', len(cz_dataset_hparams_grid))
print('Model Hyperparameters Grid Size: ', len(model_hparams_grid))
print('Total Hyperparameter Combinations: ', len(cz_dataset_hparams_grid) * len(model_hparams_grid))

Dataset Hyperparameters Grid Size:  4
Model Hyperparameters Grid Size:  1
Total Hyperparameter Combinations:  4


In [23]:
dataset_name = 'cz_dataset'
dataset_path = '../../sources/local_datasets/data/cz_dataset/cz_dataset.csv'

conduct_model_experiments(dataset_name,
                          dataset_path,
                          dataset_hparams_grid,
                          model_hparams_grid)

### Run 1 of 4

2025-05-16 00:17:02,343 - Experiment - INFO - MLflow experiment initialized with ID: 237384892919634716
2025-05-16 00:17:02,344 - Experiment - INFO - Model signature: model(mn=bert,ti=bert-base-uncased,mmn=loss,e=10,bs=16,ebs=16,lr=5e-05,ls=linear,wd=0.001,o=adamw_torch,esp=3,est=0.01)
2025-05-16 00:17:02,344 - Experiment - INFO - Dataset signature: dataset(dn=cz_dataset,tp=0.7,vp=0.15);pipeline([truncation(f=0.05,tt=top),bert_base_uncased_encoder(t=bert-base-uncased,t=true,p=max_length,tml=128,isiw=false)])
2025-05-16 00:17:02,345 - Experiment - INFO - Run signature: model(mn=bert,ti=bert-base-uncased,mmn=loss,e=10,bs=16,ebs=16,lr=5e-05,ls=linear,wd=0.001,o=adamw_torch,esp=3,est=0.01);dataset(dn=cz_dataset,tp=0.7,vp=0.15);pipeline([truncation(f=0.05,tt=top),bert_base_uncased_encoder(t=bert-base-uncased,t=true,p=max_length,tml=128,isiw=false)])
2025-05-16 00:17:03,320 - Experiment - INFO - Run ID: 480312c7e8f144d88d5d5f824f019cac
2025-05-16 00:17:03,321 - Experiment - INFO - Run name: 

Map:   0%|          | 0/34 [00:00<?, ? examples/s]

Map:   0%|          | 0/34 [00:00<?, ? examples/s]

Creating CSV from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

...

2025-05-16 00:18:34,493 - Experiment - INFO - Test metrics: {'test_loss': 0.644078254699707, 'test_accuracy': 0.7142857142857143, 'test_precision': 0.6666666666666666, 'test_recall': 0.6666666666666666, 'test_f1_score': 0.6666666666666666, 'test_roc_auc': 0.75, 'test_false_positives_rate': 0.25, 'test_false_negatives_rate': 0.3333333333333333, 'test_runtime': 0.0746, 'test_samples_per_second': 93.775, 'test_steps_per_second': 13.396, 'test_epoch': 4.0}
2025-05-16 00:18:34,519 - Experiment - INFO - Saving evaluation data in pickle...
2025-05-16 00:18:34,521 - Experiment - INFO - Saving evaluation data in json...
2025-05-16 00:18:34,552 - Experiment - INFO - Successfully saved evaluation data.
2025-05-16 00:18:34,554 - Experiment - INFO - Evaluating model using f1_score metric...
2025-05-16 00:18:34,555 - Experiment - INFO - Found checkpoint with best metrics at: checkpoint-12
2025-05-16 00:18:35,059 - Experiment - INFO - Best entry according to validation metrics: {'eval_loss': 0.608106

2025-05-16 00:18:35,148 - Experiment - INFO - Test metrics: {'test_loss': 0.644078254699707, 'test_accuracy': 0.7142857142857143, 'test_precision': 0.6666666666666666, 'test_recall': 0.6666666666666666, 'test_f1_score': 0.6666666666666666, 'test_roc_auc': 0.75, 'test_false_positives_rate': 0.25, 'test_false_negatives_rate': 0.3333333333333333, 'test_runtime': 0.0829, 'test_samples_per_second': 84.422, 'test_steps_per_second': 12.06, 'test_epoch': 4.0}
2025-05-16 00:18:35,163 - Experiment - INFO - Saving evaluation data in pickle...
2025-05-16 00:18:35,165 - Experiment - INFO - Saving evaluation data in json...
2025-05-16 00:18:35,187 - Experiment - INFO - Successfully saved evaluation data.
2025-05-16 00:18:35,188 - Experiment - INFO - Evaluating model using false_positives_rate metric...
2025-05-16 00:18:35,189 - Experiment - INFO - Found checkpoint with best metrics at: checkpoint-12
2025-05-16 00:18:35,581 - Experiment - INFO - Best entry according to validation metrics: {'eval_loss

2025-05-16 00:18:35,660 - Experiment - INFO - Test metrics: {'test_loss': 0.644078254699707, 'test_accuracy': 0.7142857142857143, 'test_precision': 0.6666666666666666, 'test_recall': 0.6666666666666666, 'test_f1_score': 0.6666666666666666, 'test_roc_auc': 0.75, 'test_false_positives_rate': 0.25, 'test_false_negatives_rate': 0.3333333333333333, 'test_runtime': 0.0742, 'test_samples_per_second': 94.374, 'test_steps_per_second': 13.482, 'test_epoch': 4.0}
2025-05-16 00:18:35,678 - Experiment - INFO - Saving evaluation data in pickle...
2025-05-16 00:18:35,679 - Experiment - INFO - Saving evaluation data in json...
2025-05-16 00:18:35,700 - Experiment - INFO - Successfully saved evaluation data.
2025-05-16 00:18:35,701 - Experiment - INFO - Evaluating model using roc_auc metric...
2025-05-16 00:18:35,702 - Experiment - INFO - Found checkpoint with best metrics at: checkpoint-12
2025-05-16 00:18:36,068 - Experiment - INFO - Best entry according to validation metrics: {'eval_loss': 0.6081066

2025-05-16 00:18:36,159 - Experiment - INFO - Test metrics: {'test_loss': 0.644078254699707, 'test_accuracy': 0.7142857142857143, 'test_precision': 0.6666666666666666, 'test_recall': 0.6666666666666666, 'test_f1_score': 0.6666666666666666, 'test_roc_auc': 0.75, 'test_false_positives_rate': 0.25, 'test_false_negatives_rate': 0.3333333333333333, 'test_runtime': 0.0838, 'test_samples_per_second': 83.56, 'test_steps_per_second': 11.937, 'test_epoch': 4.0}
2025-05-16 00:18:36,189 - Experiment - INFO - Saving evaluation data in pickle...
2025-05-16 00:18:36,193 - Experiment - INFO - Saving evaluation data in json...
2025-05-16 00:18:36,226 - Experiment - INFO - Successfully saved evaluation data.
2025-05-16 00:18:36,227 - Experiment - INFO - Evaluating model using loss metric...
2025-05-16 00:18:36,228 - Experiment - INFO - Found checkpoint with best metrics at: checkpoint-3
2025-05-16 00:18:36,666 - Experiment - INFO - Best entry according to validation metrics: {'eval_loss': 0.601536810398

2025-05-16 00:18:36,747 - Experiment - INFO - Test metrics: {'test_loss': 0.7971214652061462, 'test_accuracy': 0.42857142857142855, 'test_precision': 0.42857142857142855, 'test_recall': 1.0, 'test_f1_score': 0.6, 'test_roc_auc': 0.41666666666666663, 'test_false_positives_rate': 1.0, 'test_false_negatives_rate': 0.0, 'test_runtime': 0.0746, 'test_samples_per_second': 93.883, 'test_steps_per_second': 13.412, 'test_epoch': 1.0}
2025-05-16 00:18:36,767 - Experiment - INFO - Saving evaluation data in pickle...
2025-05-16 00:18:36,769 - Experiment - INFO - Saving evaluation data in json...
2025-05-16 00:18:36,791 - Experiment - INFO - Successfully saved evaluation data.
2025-05-16 00:18:36,792 - Experiment - INFO - Finished model evaluations stage.
